# 情報論A 第5回

自動対応付けと最小二乗法「パノラマ画像を作ってみよう その2」

###準備

`samples`に入っている`pano_ref.jpg`と`pano_src.jpg`をcolabに投げ込んでください

（あるいは、Googleドライブを設定している場合、次のセルでコメントを解除し、Googleドライブをマウントしてください）

In [ ]:
import cv2
import numpy as np  # PythonのOpenCVでは、画像はnumpyのarrayとして管理される
from google.colab.patches import cv2_imshow # colab内で画像表示関数がうまく動かないので、パッチが提供されている

# Googleドライブをマウントする場合
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/My Drive/Colab Notebooks/johoronA/"

In [ ]:
# imgをrefに張り合わせることを考える
ref = cv2.imread("pano_ref.jpg") # ベースとなる画像（BGR）
src = cv2.imread("pano_src.jpg") # 変換する画像（BGR）

# 参考： https://docs.opencv.org/master/db/d27/tutorial_py_table_of_contents_feature2d.html

# SIFT を使う場合の例。AKAZEならAKAZE_create、ORBならORB_create
sift = cv2.SIFT_create()

# 各画像に対するkeypointとdescriptorの計算
kp_ref, des_ref = sift.detectAndCompute(ref, None)
kp_src, des_src = sift.detectAndCompute(src, None)

# マッチング。SIFTなどはユークリッド距離（cv2.NORM_L2）。ORBなどはハミング距離（cv2.NORM_HAMMING）。
matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
matches = matcher.match(des_ref,des_src)

# 距離が小さい順に並べてみる（可視化のため）
matches = sorted(matches, key = lambda x:x.distance)

# 最も小さい距離で対応づいた100組の対応点を可視化してみる
corr_disp = cv2.drawMatches(ref,kp_ref,src,kp_src,matches[:100],None,flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
cv2_imshow(corr_disp) # 表示

# 対応点の登録
dst_points = np.float32([ kp_ref[m.queryIdx].pt for m in matches]).reshape(-1,1,2)
src_points = np.float32([ kp_src[m.trainIdx].pt  for m in matches]).reshape(-1,1,2)

# OpenCVの点配列は (N, 1, 2) になっていることがあるので、(N, 2) に直す
src_points = np.asarray(src_points, dtype=float).reshape(-1, 2)
dst_points = np.asarray(dst_points, dtype=float).reshape(-1, 2)

## 課題：線形最小二乗法の実装

In [ ]:
# 以下に、ホモグラフィ行列 H を最小二乗法で求めるための関数を実装しましょう
# 第4回の実装を参考に。

# src_points: 変換前の点 (x, y)  …… src画像上の点
# dst_points: 変換後の点 (x', y') …… ref画像上の点
# 返り値: src -> ref へのホモグラフィ行列 H
# ヒント：numpyでの擬似逆行列の計算：A_pinv = np.linalg.pinv(A)

def estimate_homography_least_squares(src_points, dst_points):
    A = []
    b = []

    ...

    # 対応点が4点より多い場合、Aは縦長行列になる。
    # そのため、擬似逆行列を使って最小二乗解を求める。
    ...

    # h33 = 1.0 としてホモグラフィ行列 H を作る
    ...

    return H

In [ ]:
# まずは、全対応点をそのまま使った最小二乗法の結果を見てみる
# 外れ値が混ざっているので、この結果はあまり良くない可能性がある
H_all = estimate_homography_least_squares(src_points, dst_points)
print("H_all =")
print(H_all)

# 逆変換によるパノラマスティッチング（変更不要）
dst = np.zeros((src.shape[0],src.shape[1]*2,3), dtype=np.uint8) # 横幅2倍の画像を生成(縦src.shape[0],横src.shape[1],3ch)
dst[0:ref.shape[0], 0:ref.shape[1], :] = ref # 最初に、refの画素値を先に入れておく（部分配列の操作）。3次元目は色なので、そのまま(:)
H_inv = np.linalg.inv(H_all)  # 逆変換にしたいので、逆行列を求める

for dst_y in range(dst.shape[0]):
  for dst_x in range(dst.shape[1]):
    dst_xyw = np.float32([dst_x, dst_y, 1]) # 同次座標
    src_xyw = H_inv.dot(dst_xyw)  # 変換
    src_x = src_xyw[0]/src_xyw[2] # 出力画像のX
    src_y = src_xyw[1]/src_xyw[2] # 出力画像のY

    if src_x < 1 or src_y < 1 or src_x > src.shape[1]-2 or src_y > src.shape[0]-2:  # 画像の外側を参照しないようにする
      continue

    dst[dst_y][dst_x] = src[int(src_y+0.5)][int(src_x+0.5)] # 最近傍法による補間


cv2_imshow(dst) # 表示

## RANSACによる外れ値除去（変更不要）

In [ ]:
# RANSAC+最小二乗法によるホモグラフィ行列の計算（変更不要）

# RANSACのパラメータ
threshold = 5.0      # 何画素以内なら「当たり」の対応点とみなすか
num_iter = 2000      # ランダムに4点を選ぶ試行回数
rng = np.random.default_rng(0)  # 結果を再現しやすくするため乱数シードを固定

best_inliers = None
best_num_inliers = 0
best_error = np.inf

for i in range(num_iter):
    # ホモグラフィ行列の計算には最低4組の対応点が必要
    sample_idx = rng.choice(len(src_points), 4, replace=False)

    sample_src = src_points[sample_idx]
    sample_dst = dst_points[sample_idx]

    # ランダムに選んだ4点から仮のホモグラフィ行列を計算
    H_tmp = estimate_homography_least_squares(sample_src, sample_dst)

    # src側の全点を H_tmp で ref側へ射影する
    src_h = np.hstack([src_points, np.ones((len(src_points), 1))])  # 同次座標にする
    proj_h = (H_tmp @ src_h.T).T
    proj_xy = proj_h[:, :2] / proj_h[:, 2:3]

    # 射影された点と、実際のref側対応点との距離を計算
    errors = np.linalg.norm(proj_xy - dst_points, axis=1)

    # 距離がthreshold以下の点をinlierとする
    inliers = errors < threshold
    num_inliers = np.sum(inliers)
    mean_error = np.mean(errors[inliers]) if num_inliers > 0 else np.inf

    # inlier数が多いものを採用。
    # inlier数が同じなら、平均誤差が小さいものを採用。
    if (num_inliers > best_num_inliers) or (num_inliers == best_num_inliers and mean_error < best_error):
        best_inliers = inliers
        best_num_inliers = num_inliers
        best_error = mean_error

print("inliers:", best_num_inliers, "/", len(src_points))
print("mean reprojection error:", best_error)

# RANSACで得られたinlierだけを使って、最後にもう一度、最小二乗法でHを推定する
H = estimate_homography_least_squares(src_points[best_inliers], dst_points[best_inliers])

print("H =")
print(H)


In [ ]:
##検算　おおまかに大体一致すれば良い
H_opencv, mask = cv2.findHomography(src_points, dst_points, cv2.RANSAC,5.0) # OpenCVによるホモグラフィ行列の推定（img -> refへの変換 w/ RANSAC。閾値は5.0画素にした）
print(H_opencv)

In [ ]:
# 逆変換によるパノラマスティッチング（変更不要）
dst = np.zeros((src.shape[0],src.shape[1]*2,3), dtype=np.uint8) # 横幅2倍の画像を生成(縦src.shape[0],横src.shape[1],3ch)
dst[0:ref.shape[0], 0:ref.shape[1], :] = ref # 最初に、refの画素値を先に入れておく（部分配列の操作）。3次元目は色なので、そのまま(:)
H_inv = np.linalg.inv(H)  # 逆変換にしたいので、逆行列を求める

for dst_y in range(dst.shape[0]):
  for dst_x in range(dst.shape[1]):
    dst_xyw = np.float32([dst_x, dst_y, 1]) # 同次座標
    src_xyw = H_inv.dot(dst_xyw)  # 変換
    src_x = src_xyw[0]/src_xyw[2] # 出力画像のX
    src_y = src_xyw[1]/src_xyw[2] # 出力画像のY

    if src_x < 1 or src_y < 1 or src_x > src.shape[1]-2 or src_y > src.shape[0]-2:  # 画像の外側を参照しないようにする
      continue

    dst[dst_y][dst_x] = src[int(src_y+0.5)][int(src_x+0.5)] # 最近傍法による補間


cv2_imshow(dst) # 表示